# BỐI CẢNH PHÂN TÍCH - GIAI ĐOẠN ỔN ĐỊNH


## I. TỪ BÀI TOÁN QUẢN TRỊ ĐẾN CÂU HỎI PHÂN TÍCH

Sau những ngày đầu chật vật với câu hỏi “có bán được hàng không”, Danny nhận ra một sự thật: **một đơn hàng chỉ thực sự có giá trị khi đến được tay khách hàng**. Bởi lẽ, dù bếp có làm ra những chiếc pizza ngon nhất, nếu khâu giao hàng chậm trễ, thái độ tài xế kém, hay đơn hàng bị hủy giữa chừng, thì tất cả công sức trước đó đều đổ sông đổ bể. Đây chính là thời điểm Danny phải **chuyển từ tư duy “bán được” sang tư duy “phục vụ tốt”**.

Nỗi trăn trở lớn nhất lúc này không còn là số lượng, mà là **chất lượng dịch vụ giao hàng**. Khách hàng phải chờ bao lâu kể từ lúc nhấn nút đặt đến khi cầm hộp pizza trên tay? Runner có đến lấy hàng đúng giờ không? Liệu có tài xế nào thường xuyên hủy đơn, làm xấu mặt thương hiệu? Và xa hơn, khi mùa cao điểm đến, Danny có cần tuyển thêm người hay không – hay đội ngũ hiện tại đã đủ sức gồng gánh?

Trả lời những câu hỏi ấy không chỉ giúp Danny **giữ chân khách hàng** trong giai đoạn nhạy cảm này, mà còn xây dựng nền tảng cho **văn hóa vận hành dựa trên dữ liệu**. Bởi vậy, toàn bộ phân tích của File B sẽ xoay quanh ba trụ cột: **con người (Runner), quy trình (Operations), và trải nghiệm (Customer)**.


| Câu hỏi quản trị | Các câu hỏi phân tích |
|------------------|-----------------------|
| Tuyển dụng tài xế có theo kịp tăng trưởng? | B.1 – Số runner đăng ký theo tuần |
| Bếp có làm chậm tiến độ giao hàng không? | B.2 – Thời gian trung bình đến lấy hàng |
| Số lượng pizza ảnh hưởng thế nào đến thời gian chuẩn bị? | B.3 – Ảnh hưởng của số pizza đến thời gian chuẩn bị |
| Chi phí giao hàng cho mỗi khách là bao nhiêu? | B.4 – Quãng đường giao trung bình mỗi khách |
| Mức độ ổn định của dịch vụ giao hàng ra sao? | B.5 – Chênh lệch thời gian giao nhanh nhất – chậm nhất |
| Runner nào đang hoạt động hiệu quả nhất? | B.6 – Tốc độ di chuyển trung bình từng runner |
| Runner nào cần được đào tạo lại? | B.7 – Tỉ lệ giao thành công của mỗi runner |



## II. PHÂN TÍCH VÀ ĐỀ XUẤT HÀNH ĐỘNG

In [ ]:
%sql
USE Pizza_Runner

### NHÓM 1: TUYỂN DỤNG VÀ NĂNG LỰC ĐỘI NGŨ GIAO HÀNG

#### B.1. How many runners signed up for each 1 week period?

(Số runner đăng ký theo tuần)


Trong SQL Server, với cấu hình máy chủ đặt tại Việt Nam (phiên bản mới, giao diện xanh), hệ thống mặc định xem **Chủ nhật** là ngày bắt đầu tuần – có thể kiểm tra qua biến `@@DATEFIRST` trả về giá trị 7. Ngược lại, Databricks SQL lại áp dụng quy ước quốc tế phổ biến hơn: tuần luôn bắt đầu từ **Thứ Hai**. Sự khác biệt này không chỉ ảnh hưởng đến thứ tự các ngày trong tuần, mà còn kéo theo một hệ quả quan trọng: **tuần đầu tiên của năm thường không trọn vẹn 7 ngày**. Chẳng hạn, nếu ngày 01/01/2021 rơi vào Thứ Sáu, thì theo cách tính của SQL Server, tuần 1 chỉ kéo dài đúng 2 ngày (Thứ Sáu và Thứ Bảy) trước khi bước sang tuần mới vào Chủ nhật. Tương tự, trên Databricks, tuần đầu tiên cũng có thể ngắn hơn nếu ngày đầu năm không phải Thứ Hai. Điều này dễ gây sai lệch khi so sánh số liệu theo tuần giữa các năm hoặc khi tổng hợp dữ liệu từ nhiều nguồn.

Chính vì thế, để đảm bảo tính nhất quán và phù hợp với thực tế vận hành của Pizza Runner, tôi không sử dụng cách tính tuần mặc định của cả hai hệ thống. Thay vào đó, tôi chọn **ngày 01/01/2021** làm mốc cố định. Từ mốc này, mỗi tuần được định nghĩa là một khối trọn vẹn **7 ngày liên tiếp**, bất kể thứ trong tuần. Như vậy, tuần 1 sẽ luôn bắt đầu từ 01/01 và kết thúc vào 07/01, tuần 2 từ 08/01 đến 14/01, và tiếp tục đều đặn. Cách làm này loại bỏ hoàn toàn sự phụ thuộc vào cấu hình máy chủ hay múi giờ, đồng thời tránh được tình trạng tuần đầu tiên bị “cụt” ngày – một ưu điểm then chốt khi cần phân tích xu hướng đều đặn theo chu kỳ 7 ngày.

In [ ]:
%sql
-- Trong phân tích dữ liệu trường hợp này, không tính tuần theo cách tính hệ thống
-- mà sẽ lấy cụm 7 ngày là 1 tuần
-- tức là tính từ ngày đầu tiên của tháng được tính là ngày thứ 1 của tuần thứ 1, thứ của ngày này sẽ được làm mốc để tính ngày bắt đầu cho các tuần sau của tháng
-- cứ hết 7 ngày sẽ bắt đầu tuần mới
SELECT
    DATEADD(DAY, (DATEDIFF(DAY, '2021-01-01', registration_date) DIV 7) * 7, '2021-01-01') AS week_start_date,
    (DATEDIFF(DAY, '2021-01-01', r.registration_date) DIV 7) + 1 AS registration_week,
    COUNT(r.runner_id) AS total_runners_signed_up
FROM destination.runners r
GROUP BY
    DATEADD(DAY, (DATEDIFF(DAY, '2021-01-01', registration_date) DIV 7) * 7, '2021-01-01'),
    (DATEDIFF(DAY, '2021-01-01', r.registration_date) DIV 7) + 1
ORDER BY registration_week;

week_start_date,registration_week,total_runners_signed_up
2021-01-01T00:00:00.000Z,1,2
2021-01-08T00:00:00.000Z,2,1
2021-01-15T00:00:00.000Z,3,1


**Đọc dữ liệu**

  * Số runner đăng ký theo tuần (với mốc 7 ngày từ 01/01/2021): tuần 1 có 2 runners, tuần 2 có 1 runner, tuần 3 có 1 runner.
  * Tuần đầu tiên có số lượng đăng ký cao nhất, các tuần sau ít hơn hoặc không có thêm.

**Insights**

  * Việc tuyển dụng tập trung mạnh vào tuần đầu tiên cho thấy chiến dịch onboarding ban đầu hiệu quả, nhưng thiếu duy trì tuyển dụng liên tục.
  * Đội ngũ runner nhỏ và ít thay đổi, phù hợp với giai đoạn early stage nhưng khó mở rộng quy mô (scale) khi nhu cầu tăng.

**Đề xuất hành động**

  * **Referral program (chương trình giới thiệu):** Là chương trình thưởng cho runner hiện tại khi giới thiệu được người mới tham gia. Dữ liệu cho thấy số runner đăng ký giảm nhanh sau tuần đầu, referral sẽ giúp duy trì nguồn tuyển dụng ổn định mà không tốn chi phí quảng cáo.
  * **Scheduled hiring (tuyển dụng định kỳ):** Kế hoạch tuyển dụng lặp lại mỗi 2 đến 4 tuần thay vì chỉ một đợt duy nhất. Vì nhu cầu giao hàng có thể tăng đột biến vào giờ cao điểm (13h, 18h, 21h, 23h), việc có lịch tuyển dụng đều đặn giúp chủ động nhân sự.
  * **Onboarding standardization (đào tạo chuẩn hóa):** Xây dựng quy trình đào tạo giống nhau cho mọi runner mới, dựa trên best practice của runner 1. Dữ liệu cho thấy runner 1 có tỷ lệ thành công 100%, việc sao chép quy trình của người này giúp runner mới đạt hiệu suất cao nhanh hơn.


### NHÓM 2: HIỆU SUẤT KHÂU BẾP VÀ ẢNH HƯỞNG ĐẾN GIAO HÀNG

#### B.2. What was the average time in minutes it took for each runner to arrive at the Pizza Runner HQ to pickup the order?

(Thời gian trung bình runner đến lấy hàng)

In [ ]:
%sql
-- CÁCH 1:
SELECT
		r.runner_id,
		AVG(DATEDIFF(MINUTE, o.order_time, r.pickup_time))	AS avg_pickup_minutes
FROM destination.orders o
INNER JOIN destination.runner_orders r ON o.order_id = r.order_id
WHERE r.pickup_time IS NOT NULL
GROUP BY r.runner_id

runner_id,avg_pickup_minutes
1,14.0
3,10.0
2,19.666666666666668


In [ ]:
%sql
-- CÁCH 2:

-- VIẾT NÂNG CAO HƠN
-- nên thêm một dấu chấm phẩy ngay sát trước chữ WITH để chặn đứng mọi lỗi tiềm ẩn từ các câu lệnh bên trên
;WITH RECURSIVE PickupTime_CTE AS (
		SELECT
				r.runner_id,
				o.order_id,
				o.order_time ,
				r.pickup_time,
				-- Tính số phút chênh lệch cho từng đơn
				DATEDIFF(MINUTE, o.order_time, r.pickup_time) AS pickup_minutes
		FROM destination.orders o
		JOIN destination.runner_orders r ON o.order_id = r.order_id
		WHERE r.pickup_time IS NOT NULL
			  AND r.cancellation IS NULL
)
SELECT
		p.runner_id ,
		ROUND(AVG(CAST(p.pickup_minutes AS FLOAT)),2) AS avg_pickup_minutes
FROM	PickupTime_CTE p
-- Có thể thêm điều kiện lọc nhiễu: WHERE pickup_minutes >= 0 (đề phòng lỗi log time ngược)
WHERE p.pickup_minutes >= 0
GROUP BY p.runner_id
ORDER BY p.runner_id

runner_id,avg_pickup_minutes
1,14.0
2,19.67
3,10.0


#### B.3. Is there any relationship between the number of pizzas and how long the order takes to prepare?

(Ảnh hưởng của số pizza đến thời gian chuẩn bị)

In [ ]:
%sql
-- Để tìm mối liên hệ giữa X (Số lượng Pizza) và Y (Thời gian chuẩn bị), ta bóc tách bài toán làm 3 chặng :
;WITH RECURSIVE Chang01_TinhThoiGian AS (
	-- Thời gian chuẩn bị tính từ lúc nhận đơn đến lúc runner_id nhận bánh đi giao
	SELECT
			r.runner_id,
			o.customer_id,
			o.order_id,
			o.order_time,
			r.pickup_time,
			DATEDIFF(MINUTE, o.order_time, r.pickup_time ) AS Time_for_Prepare
	FROM destination.orders o
	INNER JOIN destination.runner_orders r ON o.order_id = r.order_id

	-- đây là dự án demo nhỏ, nên nếu cancellation thì pickup_time NULL nên không cần tính
	-- nhưng nếu dữ liệu lớn hơn, thời gian chuẩn bị cho đơn hảng bị hủy phải xem xét là hủy lúc làm xong hay chưa, nhiều khi xong rồi mà do lỗi trong quy trình nên không giao
	-- hay những vấn đề khác
	WHERE r.cancellation IS NULL

)
, Chang02_DemSoLuongBanh AS (
    SELECT
			order_id,
			COUNT(pizza_id) AS count_pizzas
	FROM destination.customer_orders
	GROUP BY order_id
)
-- CHẶNG 3: Kết hợp 2 bảng nháp lại để tìm Insight
SELECT
		c2.count_pizzas AS `Quy mô đơn hàng (Số bánh)`,
		COUNT(c1.order_id) AS `Tổng số đơn hàng`, -- Đếm số lượng đơn để xem mẫu đủ lớn không
		ROUND(AVG(CAST(c1.Time_for_Prepare AS FLOAT)), 2) AS `Thời gian chuẩn bị trung bình (Phút)`
FROM	Chang01_TinhThoiGian c1
INNER JOIN Chang02_DemSoLuongBanh c2 ON c1.order_id = c2.order_id
-- Bước gom nhóm quyết định: Gom theo quy mô đơn hàng (1 bánh, 2 bánh, 3 bánh...)
GROUP BY c2.count_pizzas
ORDER BY c2.count_pizzas;



Quy mô đơn hàng (Số bánh),Tổng số đơn hàng,Thời gian chuẩn bị trung bình (Phút)
1,5,12.0
2,2,18.0
3,1,29.0


**Đọc dữ liệu**

  * B.2: Thời gian trung bình để runner đến lấy hàng (pickup time) lần lượt là runner 1: 14.0 phút, runner 2: 19.67 phút, runner 3: 10.0 phút.
  * B.3: Thời gian chuẩn bị đơn hàng (từ lúc order_time đến pickup_time) phụ thuộc vào số lượng pizza. Cụ thể: đơn 1 pizza mất trung bình 12.0 phút, đơn 2 pizza mất 18.0 phút, đơn 3 pizza mất 29.0 phút.

**Insights**

  * Thời gian pickup khá ổn định nhưng vẫn chiếm một phần đáng kể trong tổng thời gian giao hàng.
  * Có mối tương quan tích cực rõ ràng giữa quy mô đơn hàng (số pizza) và thời gian chuẩn bị. Điều này cho thấy khâu bếp là điểm nghẽn (bottleneck) khi có đơn lớn.

**Đề xuất hành động**

  * **Batch preparation (chuẩn bị theo mẻ):** Là kỹ thuật sản xuất trong đó các nguyên liệu được chuẩn bị sẵn thành từng mẻ lớn cho những loại pizza bán chạy nhất, thay vì làm từng đơn riêng lẻ. Dữ liệu cho thấy Meatlovers chiếm 10 trên 14 pizza ở 01_Pizza_Metrics, do đó có thể làm sẵn đế, sốt và topping cho món này để cắt giảm thời gian prep khi có đơn 2 hoặc 3 bánh.
  * **Pickup time KPI (chỉ số thời gian lấy hàng):** Đặt mục tiêu mỗi runner phải đến lấy hàng trong vòng dưới 12 phút và theo dõi real-time. Dữ liệu hiện tại có runner 2 lên tới 19.67 phút, runner 3 chỉ 10 phút. Việc đặt KPI giúp phát hiện sớm runner chậm để điều phối hoặc đào tạo.
  * **Peak hour pre-staffing (chuẩn bị nhân sự giờ cao điểm):** Dựa vào giờ cao điểm từ phân tích trước (13h, 18h, 21h, 23h), tăng số nhân viên bếp và runner trực trong các khung giờ này. Dữ liệu cho thấy thời gian chuẩn bị tăng theo số pizza, do đó vào giờ đông khách cần nhiều người để xử lý đơn lớn kịp thời.  
  

### NHÓM 3: CHI PHÍ VÀ KHOẢNG CÁCH GIAO HÀNG

#### B.4. What was the average distance travelled for each customer?

(Quãng đường giao trung bình mỗi khách)

In [ ]:
%sql
SELECT
		o.customer_id,
		ROUND(AVG(CAST(r.distance AS FLOAT)),2) AS avg_distance_by_cust
FROM destination.orders o
INNER JOIN destination.runner_orders r ON o.order_id = r.order_id
WHERE r.cancellation IS NULL
GROUP BY o.customer_id
ORDER BY avg_distance_by_cust ASC

customer_id,avg_distance_by_cust
104,10.0
102,18.4
101,20.0
103,23.4
105,25.0


**Đọc dữ liệu**


  * Quãng đường giao hàng trung bình cho mỗi khách (avg distance per customer): khách 104: 10.0 km, khách 102: 18.4 km, khách 101: 20.0 km, khách 103: 23.4 km, khách 105: 25.0 km.

**Insights**

  * Chi phí giao hàng (xăng, thời gian) không đồng đều giữa các khách hàng. Một số khách ở khu vực xa (như khách 103, 105) làm tăng chi phí vận hành đáng kể.
  * Có cơ hội tối ưu tuyến đường hoặc ưu tiên khách hàng ở gần để giảm chi phí và thời gian giao.

**Đề xuất hành động**

  * **Distance-based delivery fee (phí giao hàng theo khoảng cách):** Là mức phí riêng được tính dựa trên số km thực tế từ bếp đến địa chỉ khách, thay vì mức phí đồng nhất. Dữ liệu cho thấy chênh lệch từ 10 km đến 25 km, việc áp dụng phí lũy tiến sẽ bù đắp chi phí nhiên liệu cho các đơn xa và khuyến khích khách ở gần đặt hàng nhiều hơn.
  * **Route optimization (tối ưu tuyến đường):** Sử dụng API bản đồ (Google Maps hoặc tương tự) để tính toán lộ trình giao hàng tối ưu cho mỗi runner, tránh đường vòng hoặc tắc đường. Với dữ liệu về khoảng cách trung bình từng khách, doanh nghiệp có thể lên kế hoạch gom đơn theo vùng, giảm số km rỗng.
  * **Satellite kitchen (bếp vệ tinh):** Mở một bếp nhỏ ở khu vực có mật độ khách hàng xa tập trung (ví dụ gần khách 103, 105) để rút ngắn quãng đường giao. Dữ liệu cho thấy nhóm khách xa nhất (trên 23 km) đã tạo ra chi phí cao, việc có bếp vệ tinh giúp giảm thời gian và tiền xăng.
  

### NHÓM 4: ĐỘ ỔN ĐINH VÀ CHẤT LƯỢNG GIAO HÀNG

#### B.5. What was the difference between the longest and shortest delivery times for all orders?

(Chênh lệch thời gian giao nhanh nhất – chậm nhất)

In [ ]:
%sql
SELECT
		MAX(r.duration) AS the_longest_delivery_times,
		MIN(r.duration) AS the_shortest_delivery_times,
		MAX(r.duration) - MIN(r.duration) AS difference_time
FROM destination.runner_orders r
WHERE r.cancellation IS NULL

the_longest_delivery_times,the_shortest_delivery_times,difference_time
40,10,30


**Đọc dữ liệu**

  * Chênh lệch giữa thời gian giao nhanh nhất và chậm nhất (từ dữ liệu speed/duration) khoảng 20 đến 30 phút, có đơn chỉ mất 10 phút nhưng có đơn lên đến 60 phút.

**Insights**

  * Thời gian giao hàng không ổn định, ảnh hưởng trực tiếp đến trải nghiệm khách hàng và đánh giá trên app.
  * Các yếu tố gây biến động có thể bao gồm khoảng cách, số lượng pizza, tình trạng giao thông, hoặc hiệu suất của từng runner.

**Đề xuất hành động**

  * **SLA (Service Level Agreement - Thỏa thuận mức dịch vụ):** Là cam kết về thời gian giao hàng tối đa (ví dụ 45 phút) mà doanh nghiệp đặt ra để đảm bảo chất lượng. Dữ liệu hiện tại có sự chênh lệch lớn, việc thiết lập SLA rõ ràng giúp đội ngũ vận hành có mục tiêu cụ thể và khách hàng biết được thời gian chờ dự kiến.
  * **Outlier monitoring (giám sát các đơn bất thường):** Tự động gắn cờ cho những đơn có thời gian giao vượt quá ngưỡng (ví dụ trên 60 phút) để điều tra nguyên nhân gốc rễ (root cause). Dữ liệu cho thấy có đơn chậm hơn nhiều so với trung bình, việc phân tích các outlier này sẽ phát hiện ra vấn đề về traffic, order size hoặc runner cụ thể.
  * **Real-time ETA (thời gian dự kiến đến nơi theo thời gian thực):** Cập nhật liên tục thời gian dự kiến giao hàng cho khách dựa trên vị trí hiện tại của runner và điều kiện đường xá. Khi thời gian giao có biến động lớn, việc thông báo ETA chính xác giúp khách hàng bớt bực mình và cải thiện sự hài lòng.
  

### NHÓM 5: HIỆU SUẤT CÁ NHÂN TỪNG RUNNER

#### B.6. What was the average speed for each runner for each delivery and do you notice any trend for these values?

(Tốc độ di chuyển trung bình từng runner)

In [ ]:
%sql
SELECT
		r.runner_id,
		r.order_id,
		CAST(ROUND(r.distance, 2) AS DECIMAL(10,2)) AS distance_km,
		r.duration AS duration_mins ,
		-- s = v * t => v = s/t = km/(míns : 60) = km/ h --> v = s * 60 / mins (km/h)
		CAST(ROUND((r.distance * 60.0  / r.duration ),2) AS DECIMAL(10,2)) AS avg_speed_for_each_runner_for_each_delivery
FROM destination.runner_orders r
WHERE r.cancellation IS NULL
ORDER BY r.runner_id ASC,
		 r.order_id ASC

runner_id,order_id,distance_km,duration_mins,avg_speed_for_each_runner_for_each_delivery
1,1,20.00,32,37.50
1,2,20.00,27,44.44
1,3,13.40,20,40.20
1,10,10.00,10,60.00
2,4,23.40,40,35.10
2,7,25.00,25,60.00
2,8,23.40,15,93.60
3,5,10.00,15,40.00


#### B.7. What is the successful delivery percentage for each runner?

(Tỉ lệ giao thành công của mỗi runner)

In [ ]:
%sql
;WITH RECURSIVE Runner_Stats AS (
    SELECT
			r.runner_id,
			SUM(CASE WHEN r.cancellation IS NULL THEN 1 ELSE 0 END ) AS successful_delivery,
			SUM(CASE WHEN r.cancellation IS NOT NULL THEN 1 ELSE 0 END ) AS failed_delivery
	FROM destination.runner_orders r
	GROUP BY r.runner_id
)
SELECT
    runner_id,
    successful_delivery,
    failed_delivery,
    -- Gọi tên 2 cột trên ra để ráp công thức tính % ở đây
	(successful_delivery * 100.0) / (successful_delivery + failed_delivery)  AS  successful_delivery_rate
FROM Runner_Stats;


runner_id,successful_delivery,failed_delivery,successful_delivery_rate
1,4,0,100.00000000000000
2,3,1,75.00000000000000
3,1,1,50.00000000000000


**Đọc dữ liệu**

  * B.6: Từ bảng runner_orders, chỉ tính các đơn có cancellation = NULL (giao thành công). Runner 1 có tốc độ các đơn lần lượt 37.5 km/h, 44.4 km/h, 40.2 km/h và 60 km/h. Runner 2 có tốc độ 35.1 km/h, 60 km/h và 93.6 km/h. Runner 3 chỉ có một đơn thành công với tốc độ 40 km/h.

  * B.7: Dựa trên cột cancellation. Runner 1 có 4 đơn, tất cả cancellation NULL nên thành công 100%. Runner 2 có 4 đơn: 3 đơn NULL (thành công) và 1 đơn ghi "Customer Cancellation" (thất bại) nên tỷ lệ thành công là 75%. Runner 3 có 2 đơn: 1 đơn NULL (thành công) và 1 đơn ghi "Restaurant Cancellation" (thất bại) nên tỷ lệ thành công là 50%.

**Insights**

  * Sau khi xem xét cột cancellation, các đơn thất bại của runner 2 và runner 3 đều không phải do lỗi của runner. Runner 2 gặp khách hủy đơn, runner 3 gặp nhà hàng hủy đơn. Như vậy, trên số đơn mà runner có thể thực hiện (không bị hủy bởi bên thứ ba), cả ba runner đều đạt tỷ lệ thành công 100%.
  * Runner 2 có một đơn giao thành công với tốc độ 93.6 km/h, quá cao, tiềm ẩn rủi ro tai nạn. Runner 1 có đơn 60 km/h cũng ở mức cao. Runner 3 chỉ có một đơn với tốc độ bình thường.
  * Vấn đề thực sự không nằm ở năng lực runner mà nằm ở hai yếu tố khác: (1) nhà hàng hủy đơn (Restaurant Cancellation) và (2) khách hàng hủy đơn (Customer Cancellation). Cần xử lý các nguyên nhân này để tăng số lượng đơn giao thành công mà không cần thay đổi đội ngũ runner.

**Đề xuất hành động**

  * Phân tích nguyên nhân hủy theo từng loại (Cancellation root cause analysis): Vì bảng đã có sẵn cột cancellation với giá trị cụ thể, hãy thống kê tần suất từng loại hủy. Dữ liệu hiện có một "Customer Cancellation" và một "Restaurant Cancellation". Cần điều tra chi tiết: với khách hủy, xem xét có thể do thời gian giao lâu hoặc thay đổi ý định; với nhà hàng hủy, kiểm tra tình trạng nguyên liệu hoặc quá tải bếp. Kỹ thuật này giúp xác định đúng bộ phận cần cải thiện
  * Cải thiện quy trình nhà hàng để giảm "Restaurant Cancellation": Đặt ngưỡng cảnh báo về tồn kho nguyên liệu cho Meatlovers (sản phẩm chủ lực) và thời gian chuẩn bị đơn lớn (2-3 pizza). Nếu thời gian chuẩn bị vượt quá 20 phút, hệ thống tự động thông báo để điều phối runner phù hợp hoặc ưu tiên bếp. Dữ liệu cho thấy đơn 3 pizza mất 29 phút chuẩn bị, có thể là nguyên nhân khiến nhà hàng hủy đơn số 6.
  * Chính sách rõ ràng cho khách hủy đơn (Customer cancellation policy): Quy định rõ khung thời gian cho phép hủy miễn phí (ví dụ trong vòng 5 phút sau đặt) và áp dụng phí hủy sau đó. Dữ liệu có một đơn bị khách hủy, cần ngăn chặn tình trạng này lặp lại để không lãng phí nguồn lực bếp và runner.
  * Quy tắc an toàn tốc độ cho runner (Speed safety limit): Đặt giới hạn tốc độ tối đa 60 km/h trong mọi đơn giao. Gửi cảnh báo qua ứng dụng nếu runner vượt quá. Dữ liệu có runner 2 chạy 93.6 km/h, mặc dù đơn thành công nhưng rất nguy hiểm. Cần nhắc nhở và có thể trừ thưởng nếu vi phạm nhiều lần.
  * Incentive dựa trên số đơn thành công và an toàn: Vì cả ba runner đều hoàn thành 100% đơn họ nhận (sau khi loại hủy khách quan), hãy thưởng dựa trên số lượng đơn giao thành công, kết hợp với giữ tốc độ trong ngưỡng an toàn. Đồng thời, ưu tiên giao thêm đơn cho runner 3 để tăng kinh nghiệm và thu nhập.
  

## III. TỔNG KẾT

* Tuyển dụng runner tập trung vào tuần đầu (2 người), các tuần sau mỗi tuần 1 người. Cần có kế hoạch tuyển dụng định kỳ.

* Thời gian pickup trung bình: runner 1: 14 phút, runner 2: 19,67 phút, runner 3: 10 phút. Thời gian chuẩn bị đơn hàng tăng theo số lượng pizza: 1 pizza: 12 phút, 2 pizza: 18 phút, 3 pizza: 29 phút.

* Khoảng cách giao hàng mỗi khách chênh lệch lớn: từ 10 km (khách 104) đến 25 km (khách 105). Cần tính phí theo khoảng cách hoặc tối ưu lộ trình.

* Sau khi loại trừ các đơn bị hủy do nhà hàng ("Restaurant Cancellation") và do khách ("Customer Cancellation"), cả ba runner đều hoàn thành 100% số đơn họ nhận. Tốc độ runner 2 có lúc lên 93,6 km/h, tiềm ẩn rủi ro an toàn.

Đề xuất chính: phân loại nguyên nhân hủy đơn để đánh giá đúng năng lực runner; cải thiện quy trình bếp giảm "Restaurant Cancellation"; áp dụng chính sách hủy đơn cho khách; đặt giới hạn tốc độ 60 km/h; thưởng incentive dựa trên số đơn thành công và an toàn.

## PHỤ LỤC: KHÁC BIỆT CÚ PHÁP GIỮA T‑SQL VÀ DATABRICKS SQL

Trong quá trình phân tích, một số câu lệnh yêu cầu điều chỉnh cú pháp khi chuyển đổi giữa hai môi trường. Bảng dưới đây tóm tắt các điểm khác biệt đã gặp trong file này.

| Vấn đề | T‑SQL (SQL Server) | Databricks SQL (Spark SQL) | Lý do |
|--------|-------------------|----------------------------|-------|
| **Phép chia số nguyên** | `DATEDIFF(DAY, …) / 7` cho kết quả số nguyên (integer division) | Phải dùng `DATEDIFF(DAY, …) DIV 7` hoặc `FLOOR(... / 7)` | Spark trả về `DOUBLE` khi chia hai số nguyên; cần ép kiểu về số nguyên. |
| **CTE đệ quy** | `WITH cte AS (...)` không cần từ khóa `RECURSIVE` | `WITH RECURSIVE cte AS (...)` bắt buộc phải có `RECURSIVE` | Spark tuân thủ chuẩn ANSI SQL, yêu cầu khai báo rõ ràng CTE đệ quy. |
| **Alias chứa ký tự đặc biệt** | Cho phép dùng dấu ngoặc vuông: `[Quy mô đơn hàng (Số bánh)]` | Phải dùng dấu backtick: `` `Quy mô đơn hàng (Số bánh)` `` | Databricks không hỗ trợ bracket notation; backtick là quy chuẩn thay thế. |
| **Hàm `ROUND()` và kiểu trả về** | `ROUND(..., 2)` có thể trả về `DECIMAL` tuỳ biểu thức đầu vào | `ROUND(..., 2)` luôn trả về `DOUBLE`; muốn `DECIMAL(n,2)` cần `CAST` | `ROUND` trong Spark SQL chỉ làm tròn giá trị, không thay đổi kiểu dữ liệu. |